# Load Testing a Databricks Model Serving Endpoint with Locust (CLI)

We train and deploy a simple SKLearn Linear Regression model to a Databricks Model
Serving endpoint, then **load test** it with [Locust](https://github.com/locustio/locust)
using a standalone **`locustfile.py`** run through the **Locust CLI** in headless mode.
This follows the pattern in the Locust docs
([Writing a locustfile](https://docs.locust.io/en/stable/writing-a-locustfile.html)),
using Locust's built-in `HttpUser` to send HTTP requests to the endpoint.

### Environment
- **Provider**: Azure (works on any cloud)
- **Compute**: Single-node ML instance (e.g. `Standard_D4ds_v5` / `Standard_DS3_v2`,
  Runtime **16.4 LTS ML** or newer). Load generation is I/O-bound green threads, not
  Spark, so no workers are needed. The load generator (this cluster) is separate compute
  from the serving endpoint, so client-side limits aren't misattributed to the endpoint.
- **Scale**: Deliberately low concurrency — a few steady users — to measure latency and
  throughput on a `Small` endpoint. Benchmark a single instance first, then scale up.

### Additional Resources
- [Locust](https://github.com/locustio/locust)
- [Writing a locustfile](https://docs.locust.io/en/stable/writing-a-locustfile.html)
- [Locust configuration & CLI options](https://docs.locust.io/en/stable/configuration.html)
- [Creating Serving Endpoints](https://docs.databricks.com/aws/en/machine-learning/model-serving/create-manage-serving-endpoints)
- [Querying Serving Endpoints](https://docs.databricks.com/aws/en/machine-learning/model-serving/score-custom-model-endpoints)

## Install Locust

Locust is not part of the Databricks ML runtime, so we install it and restart the
Python interpreter to pick it up.

In [ ]:
%pip install locust
dbutils.library.restartPython()

## Unity Catalog Setup

We register the model to Unity Catalog. Create a catalog + schema (skip if you already
have one). Make sure your metastore has a managed storage location configured.

- [Azure metastore setup](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/create-metastore)
- [AWS metastore setup](https://docs.databricks.com/aws/en/data-governance/unity-catalog/create-metastore)

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS ram_artifacts;

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS ram_artifacts.models;

## Train & Register a Sample SKLearn Model

A small, deterministic Linear Regression model on mock data. We register it to Unity
Catalog with an input example logged so the endpoint knows the expected schema.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

mlflow.set_registry_uri("databricks-uc")

# Mock regression data
rng = np.random.default_rng(42)
n = 300
X = pd.DataFrame({
    "feature_1": rng.normal(0.0, 1.0, n),
    "feature_2": rng.normal(5.0, 2.0, n),
    "feature_3": rng.integers(0, 10, n).astype(float),
})
y = (3.0 * X["feature_1"] - 1.2 * X["feature_2"] + 0.7 * X["feature_3"]
     + rng.normal(0, 0.5, n)).to_frame(name="target")

train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
catalog = "ram_artifacts"
schema = "models"
model_name = "LinearRegressionLoadTest"
registered_name = f"{catalog}.{schema}.{model_name}"

mlflow.sklearn.autolog(log_input_examples=True, registered_model_name=registered_name)

with mlflow.start_run(run_name="lr_loadtest_train"):
    lr = LinearRegression()
    lr.fit(train_x, train_y)
    preds = lr.predict(test_x)
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(test_y, preds)))
    mlflow.log_metric("r2", r2_score(test_y, preds))

print(f"Registered UC model: {registered_name}")

## Deploy the Model Serving Endpoint

We deploy the registered model to a `Small` CPU endpoint. Adjust `workload_size` /
`scale_to_zero` for your needs.

In [ ]:
from mlflow.deployments import get_deploy_client

mlflow.set_registry_uri("databricks-uc")
client = get_deploy_client("databricks")

endpoint_name = "sklearn-loadtest-ep"
version = "1"

endpoint = client.create_endpoint(
    name=endpoint_name,
    config={
        "served_entities": [
            {
                "name": f"{model_name}-{version}",
                "entity_name": f"{catalog}.{schema}.{model_name}",
                "entity_version": version,
                "workload_size": "Small",
                "workload_type": "CPU",
                "scale_to_zero_enabled": False,
            }
        ]
    },
)

Endpoint creation takes a few minutes. Poll until it reports `READY`.

In [ ]:
import time

while True:
    ep = client.get_endpoint(endpoint_name)
    state = ep["state"]["ready"]
    print(f"Endpoint state: {state}")
    if state == "READY":
        print("✅ Endpoint is ready")
        break
    if state == "FAILED":
        raise RuntimeError(f"❌ Endpoint creation failed: {ep}")
    time.sleep(30)

## Sanity Check: Single Prediction

Confirm the endpoint scores correctly before we load test it.

In [ ]:
response = client.predict(
    endpoint=endpoint_name,
    inputs={
        "dataframe_split": {
            "columns": ["feature_1", "feature_2", "feature_3"],
            "data": [[0.15, 3.8, 7.0]],
        }
    },
)
print(response)

## The Locustfile

We write a standalone `locustfile.py` — the artifact the Locust CLI loads. It defines a
single `HttpUser` with one `@task` (`score`) that POSTs the scoring payload:

- Auth is set once in `on_start()` on the session, not per request.
- `name="score"` groups all invocations under one stats entry.
- Any 4xx/5xx is recorded as a failure automatically.
- The token and endpoint name come from environment variables; the host is passed via
  `--host`, so nothing sensitive is hard-coded.

It is written to the driver's working directory so the CLI can find it.

In [ ]:
import os

locustfile_content = r"""# Locust load test for a Databricks Model Serving endpoint.
# Config comes from env vars (DATABRICKS_TOKEN, SERVING_ENDPOINT_NAME);
# the workspace host is passed via the --host CLI flag.

import os

from locust import HttpUser, task, between

TOKEN = os.environ["DATABRICKS_TOKEN"]
ENDPOINT_NAME = os.environ["SERVING_ENDPOINT_NAME"]
INVOCATION_PATH = f"/serving-endpoints/{ENDPOINT_NAME}/invocations"

# Scoring payload matching the model's input schema.
PAYLOAD = {
    "dataframe_split": {
        "columns": ["feature_1", "feature_2", "feature_3"],
        "data": [[0.15, 3.8, 7.0]],
    }
}


class ServingUser(HttpUser):
    wait_time = between(0.1, 0.5)  # think-time between requests per user

    def on_start(self):
        # Set the auth header once on the session.
        self.client.headers.update({"Authorization": f"Bearer {TOKEN}"})

    @task
    def score(self):
        self.client.post(INVOCATION_PATH, json=PAYLOAD, name="score")
"""

with open("locustfile.py", "w") as f:
    f.write(locustfile_content)

print("Wrote locustfile.py to", os.path.abspath("locustfile.py"))

## Run the Load Test via the Locust CLI

Grab the workspace URL + a context token, export the config the locustfile expects, and
invoke Locust **headless** as a subprocess (`python -m locust`). We stream its output
line by line, so Locust's periodic stats table (printed every ~2s) shows up **live** in
the cell while the test runs — not just at the end.

Flags:

- `--headless` — no web UI, start immediately
- `-u` / `-r` — peak users / spawn rate (users per second)
- `-t 30s` — run duration
- `--host` — workspace base URL (relative paths resolve against this)
- `--csv locust_results` — writes `locust_results_stats.csv`, `_stats_history.csv`, `_failures.csv`

We omit `--only-summary` so Locust emits the periodic table as it runs.

Keep concurrency low here. Bump `USERS` / `RUN_TIME` for a heavier test.

In [ ]:
import os
import sys
import subprocess

# Workspace URL + short-lived API token from the notebook context.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
WORKSPACE_URL = ctx.apiUrl().get()          # e.g. https://adb-xxxx.azuredatabricks.net
TOKEN = ctx.apiToken().get()

USERS = 3            # low, steady concurrency
SPAWN_RATE = 1
RUN_TIME = "30s"
CSV_PREFIX = "locust_results"

# Config the locustfile reads from the environment.
env = os.environ.copy()
env["DATABRICKS_TOKEN"] = TOKEN
env["SERVING_ENDPOINT_NAME"] = endpoint_name

cmd = [
    sys.executable, "-m", "locust",
    "-f", "locustfile.py",
    "--headless",
    "-u", str(USERS),
    "-r", str(SPAWN_RATE),
    "-t", RUN_TIME,
    "--host", WORKSPACE_URL,
    "--csv", CSV_PREFIX,
]

# Stream output live: Popen + line iteration prints Locust's periodic stats
# (emitted to stderr every ~2s) as the test runs, instead of buffering to the end.
print("Running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(
    cmd, env=env, text=True, bufsize=1,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
for line in proc.stdout:
    print(line, end="")
returncode = proc.wait()
if returncode != 0:
    # Non-zero exit means at least one request failed.
    print(f"(locust exited with code {returncode})")

## Results

Locust wrote the aggregated stats to `<prefix>_stats.csv`. The `Aggregated` row holds
totals across all request types. Request counts, RPS, and avg/min/max latency are exact;
percentile columns come from Locust's response-time histogram (rounded to ~2 significant
figures — exact under 100ms).

In [ ]:
import pandas as pd

stats = pd.read_csv(f"{CSV_PREFIX}_stats.csv")
agg = stats[stats["Name"] == "Aggregated"].iloc[0]

print("==== Load Test Summary ====")
print(f"Total requests     : {int(agg['Request Count'])}")
print(f"Failures           : {int(agg['Failure Count'])}")
print(f"Requests/sec       : {agg['Requests/s']:.2f}")
print()
print("Response times (ms):")
print(f"  min              : {agg['Min Response Time']:.0f}")
print(f"  avg              : {agg['Average Response Time']:.1f}")
print(f"  median (p50)     : {agg['Median Response Time']:.0f}")
print(f"  p95              : {agg['95%']:.0f}")
print(f"  p99              : {agg['99%']:.0f}")
print(f"  max              : {agg['Max Response Time']:.0f}")

## Track Latency & TPS in MLflow

Log the config as **params** and the measured latency / throughput as **metrics**, so
runs are versioned and comparable (e.g. before vs. after a `workload_size` change). This
reads the finished CSV, so it has no effect on the measured numbers.

In [ ]:
total_requests = int(agg["Request Count"])
failures = int(agg["Failure Count"])
fail_ratio = failures / total_requests if total_requests else 0.0
rps = float(agg["Requests/s"])            # all requests / sec
tps = rps * (1 - fail_ratio)              # successful requests / sec

with mlflow.start_run(run_name="locust_loadtest"):
    mlflow.log_params({
        "endpoint_name": endpoint_name,
        "model": registered_name,
        "users": USERS,
        "spawn_rate": SPAWN_RATE,
        "run_time": RUN_TIME,
    })
    mlflow.log_metrics({
        "total_requests": total_requests,
        "failures": failures,
        "fail_ratio": fail_ratio,
        "throughput_rps": rps,
        "throughput_tps": tps,
        "latency_avg_ms": float(agg["Average Response Time"]),
        "latency_p50_ms": float(agg["Median Response Time"]),
        "latency_p95_ms": float(agg["95%"]),
        "latency_p99_ms": float(agg["99%"]),
        "latency_max_ms": float(agg["Max Response Time"]),
    })
    # Keep the raw Locust CSVs attached to the run for the full record.
    mlflow.log_artifact(f"{CSV_PREFIX}_stats.csv")

print(f"Logged load-test metrics to MLflow (TPS={tps:.2f}, p95={agg['95%']:.0f}ms)")

## Cleanup (Optional)

Delete the endpoint when you're done to stop incurring cost.

In [ ]:
client.delete_endpoint(endpoint_name)
print(f"Deleted endpoint: {endpoint_name}")